In [ ]:
import os
import pickle
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

raw_root = Path(os.environ["GAITLU_RAW_ROOT"])
output_root = Path(os.environ["GAITLU_OUTPUT_ROOT"])

shard_name = os.environ.get("GAITLU_SHARD", "gaitlu-000")

# Support .tar, .tar.gz, and .tgz.
candidates = [
    raw_root / f"{shard_name}.tar",
    raw_root / f"{shard_name}.tar.gz",
    raw_root / f"{shard_name}.tgz",
]

tar_path = next((path for path in candidates if path.is_file()), None)

if tar_path is None:
    matches = sorted(raw_root.glob(f"{shard_name}*.tar*"))
    if matches:
        tar_path = matches[0]
    else:
        raise FileNotFoundError(
            f"No shard found for {shard_name!r} under {raw_root}"
        )

print(f"Reading: {tar_path}")


def decode_sequence(value):
    if isinstance(value, dict):
        value = value.get(
            "silhouettes",
            value.get("frames", value.get("data")),
        )

    if isinstance(value, (list, tuple)) and len(value) == 1:
        value = value[0]

    frames = np.asarray(value)

    if frames.ndim == 4 and frames.shape[1] == 1:
        frames = frames[:, 0]
    elif frames.ndim == 4 and frames.shape[-1] == 1:
        frames = frames[..., 0]

    if frames.ndim != 3:
        raise ValueError(f"Expected [T, H, W], received {frames.shape}")

    if frames.max() <= 1:
        return frames >= 0.5

    return frames >= 128


with tarfile.open(tar_path, "r:*") as archive:
    pkl_members = [
        member
        for member in archive.getmembers()
        if member.isfile() and member.name.endswith(".pkl")
    ]

    if not pkl_members:
        raise RuntimeError(
            "This shard contains no .pkl files. "
            "It may be a bit-packed shard containing records/*.bits."
        )

    selected_members = pkl_members[:4]
    videos = []

    for member in selected_members:
        with archive.extractfile(member) as handle:
            # Only do this for trusted GaitLU dataset files.
            value = pickle.load(handle)

        videos.append((member.name, decode_sequence(value)))


frames_to_show = 8

fig, axes = plt.subplots(
    len(videos),
    frames_to_show,
    figsize=(16, 2.8 * len(videos)),
    squeeze=False,
)

for row_index, (member_name, video) in enumerate(videos):
    frame_indices = np.linspace(
        0,
        len(video) - 1,
        frames_to_show,
    ).astype(int)

    for column_index, frame_index in enumerate(frame_indices):
        ax = axes[row_index, column_index]
        ax.imshow(
            video[frame_index],
            cmap="gray",
            vmin=0,
            vmax=1,
            interpolation="nearest",
        )
        ax.set_title(f"frame {frame_index}")
        ax.axis("off")

    axes[row_index, 0].set_ylabel(
        Path(member_name).parent.name,
        rotation=0,
        labelpad=35,
        va="center",
    )

fig.suptitle(f"{shard_name}: sample GaitLU silhouettes")
plt.tight_layout()

output_root.mkdir(parents=True, exist_ok=True)
png_path = output_root / f"{shard_name}.png"
fig.savefig(png_path, dpi=150, bbox_inches="tight")

print(f"Saved: {png_path}")
plt.show()